# DataMate: chạy Qwen3.5-9B trên Colab

Trước khi chạy: tải `datamate.zip` (dữ liệu, chỉ cần một lần) và `datamate_code.zip` mới nhất (`python scripts/pack.py code`) vào thư mục `zaloai` trên Google Drive. Runtime: **A100** (Qwen3.5-9B bf16 ~19 GB; L4 24 GB cũng chạy được nhưng chậm). Chạy các ô theo thứ tự.

In [ ]:
# 1. Cài đặt: Drive, dữ liệu + code, Python 3.12 + vLLM 0.17.1 (Qwen3.5 cần vLLM >= 0.17). Model tải ngầm cùng lúc.
#    Giải nén bằng Python: unzip của Colab làm hỏng tên file tiếng Việt (vd "Danh sách hàng hóa.xlsx" của dev_001).
from google.colab import drive
drive.mount('/content/drive')
import glob, os, shutil, zipfile
os.chdir('/content')  # không đứng trong /content/dm khi xoá nó
Z = sorted(glob.glob('/content/drive/MyDrive/datamate.zip') + glob.glob('/content/drive/MyDrive/*/datamate.zip'))[0]
shutil.rmtree('/content/dm', ignore_errors=True)
zipfile.ZipFile(Z).extractall('/content/dm'); print('data from', Z)
C = max(glob.glob('/content/drive/MyDrive/zaloai/datamate_code*.zip'), key=os.path.getmtime)
zipfile.ZipFile(C).extractall('/content/dm'); print('code from', C, os.path.getsize(C), 'bytes')
import subprocess
dl = subprocess.Popen(['python', '-c', "from huggingface_hub import snapshot_download; snapshot_download('Qwen/Qwen3.5-9B', local_dir='/content/model')"], stdout=open('/content/dl.log', 'w'), stderr=subprocess.STDOUT)
!pip -q install uv
!uv venv -q -p 3.12 /content/venv && uv pip install -q -p /content/venv/bin/python vllm==0.17.1 pandas==2.2.3 python-calamine==0.9.1 2>&1 | tail -3
!/content/venv/bin/python -c "import vllm, transformers, sys; print('vllm', vllm.__version__, 'transformers', transformers.__version__, 'python', sys.version.split()[0])"
dl.wait(); print('download rc', dl.returncode)
!du -sh /content/model

In [ ]:
# 2. Bật model server. --language-model-only: bỏ phần xử lý ảnh, nhường bộ nhớ cho KV cache. 32768 token như predict.sh.
import subprocess, time, urllib.request
PY = '/content/venv/bin/python'
try:  # chạy lại ô này: tắt server cũ trước
    srv.terminate(); srv.wait(120)
except NameError:
    pass
srv = subprocess.Popen([PY, '-m', 'vllm.entrypoints.openai.api_server', '--model', '/content/model', '--served-model-name', 'model',
                        '--port', '8000', '--host', '127.0.0.1', '--max-model-len', '32768', '--seed', '0',
                        '--gpu-memory-utilization', '0.92', '--reasoning-parser', 'qwen3', '--language-model-only'],
                       stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)
t0 = time.time()
while True:
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=3); break
    except Exception:
        assert srv.poll() is None, open('/content/vllm.log').read()[-3000:]
        time.sleep(5)
print(f'model san sang sau {time.time() - t0:.0f} giay')

## Holdout train rồi public v7

Holdout300 (train, có đáp án) để bắt hồi quy, mốc 0,95; rồi public đủ 1000 câu. Theo dõi: Terminal → `tail -f /content/hold.log` rồi `tail -f /content/pub.log`. Kết quả chép sang Drive `zaloai/public_run_v7/`.

Dòng `could not close` phải in 0: nếu khác 0 thì vLLM không cho viết tiếp lượt trả lời, báo lại mình.

In [ ]:
# 3. Holdout300 (~12 phút) rồi public 1000 câu (~1 giờ). Ô này tự lấy datamate_code*.zip mới nhất trên Drive.
#    KHÔNG mở /content/pub/submission.jsonl trong lúc đang chạy.
import glob, os, zipfile
os.chdir('/content')
C = max(glob.glob('/content/drive/MyDrive/zaloai/datamate_code*.zip'), key=os.path.getmtime)
zipfile.ZipFile(C).extractall('/content/dm'); print('code from', C, os.path.getsize(C), 'bytes')
%cd /content/dm
!grep -c 'def conclude' src/agent.py
PY = '/content/venv/bin/python'
!rm -rf /content/hold && {PY} -X utf8 -m src.agent --data data/train/train --questions training_code/splits/holdout300.jsonl --out /content/hold --workers 16 > /content/hold.log 2>&1
!echo could not close: $(grep -c 'could not close' /content/hold.log)
!{PY} scripts/score.py /content/hold/submission.jsonl --questions training_code/splits/holdout300.jsonl | grep 'PA ='
!rm -rf /content/pub && {PY} -X utf8 -m src.agent --data data/public_test/public_test --out /content/pub --workers 16 > /content/pub.log 2>&1
!tail -2 /content/pub.log; echo could not close: $(grep -c 'could not close' /content/pub.log)
!{PY} scripts/check_submission.py /content/pub/submission.jsonl data/public_test/public_test/questions.jsonl
!mkdir -p /content/drive/MyDrive/zaloai/public_run_v7 && cp -r /content/pub/submission.jsonl /content/pub/logs /content/drive/MyDrive/zaloai/public_run_v7/ && cp /content/hold/submission.jsonl /content/drive/MyDrive/zaloai/public_run_v7/holdout_submission.jsonl && cp /content/hold/logs/log.jsonl /content/drive/MyDrive/zaloai/public_run_v7/holdout_log.jsonl && ls -la /content/drive/MyDrive/zaloai/public_run_v7

In [ ]:
srv.terminate()  # tắt model; nhớ Runtime -> Disconnect and delete runtime để khỏi tốn unit